# Wire Wheels Scraper (`bbwheels.ipynb`)

Scrapes **Wire Wheels only** from:

1. https://luxorwirewheels.com/
2. https://www.bbwheelsonline.com/

---

## Analysis of the original notebook

The original file had **2 cells**:

| Cell | Role | Stack |
|------|------|--------|
| 0 | Paginate listing pages and write `products.csv` (name, price, link) | **Selenium** on `bbwheelsonline.com/tires/?page=N` using Searchspring selectors `li.ss__result.product` |
| 1 | Open each product URL from CSV and extract SKU/brand/prices/`dl.productView-info` details | **requests + BeautifulSoup** → `bbwheels.csv` |

### What worked well (kept)

- Two-phase pipeline (collect links → enrich product pages)
- BigCommerce product selectors (`productView-title`, `productView-brand`, `price--withoutTax`, `dl.productView-info`)
- Per-product `try/except` so one failure does not stop the run
- Simple CSV output

### Gaps vs this project

- Targeted **tires**, not wire wheels
- Hard-coded tire attribute columns
- No Luxor support
- No image download / `images/brand/sku/` layout
- No retries / sessions / duplicate SKU handling
- Selenium used even when HTML APIs are available

---

## Changes in this update

1. **Reuse** the BB product-page parser pattern (Cell 1) and expand fields for wheels.
2. **Add Luxor** via Shopify `/products.json` (no browser needed).
3. **BB discovery** via XML product sitemap + wire-wheel verification (Selenium optional for Searchspring search).
4. **Images** downloaded to `images/<brand>/<sku>/imageN.jpg` with relative paths in CSV.
5. **Outputs**: `luxor_wire_wheels.csv`, `bbwheels_wire_wheels.csv`.
6. Shared helpers live in `scrape_wire_wheels.py` (imported below) so the notebook stays thin.

> Finding: BB Wheels Online currently appears to have **few/no classic wire-wheel SKUs** in its catalog. The BB CSV may be empty; Luxor is the primary source.


In [ ]:
# Setup — run this first
from pathlib import Path
import sys

# Ensure we import the local scraper module
ROOT = Path.cwd()
if ROOT.name != "wire-wheels-scraper":
    candidate = ROOT / "wire-wheels-scraper"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)
print("Module path OK:", (ROOT / "scrape_wire_wheels.py").exists())


In [ ]:
# Imports + shared session (improved version of original Cell 0/1 setup)
from scrape_wire_wheels import (
    make_session,
    scrape_luxor,
    scrape_bbwheels,
    OUTPUT_LUXOR,
    OUTPUT_BB,
    IMAGES_DIR,
)

session = make_session()
print("Session ready")
print("Images dir:", IMAGES_DIR)
print("Luxor CSV:", OUTPUT_LUXOR)
print("BB CSV:", OUTPUT_BB)


In [ ]:
# ---------------------------------------------------------------------------
# LUXOR — scrape all Wire Wheels
# Uses Shopify products.json (requests only; no Selenium)
# Set LIMIT_LUXOR = 5 for a quick smoke test, or None for full catalog
# ---------------------------------------------------------------------------
LIMIT_LUXOR = None  # e.g. 5 for testing

luxor_csv = scrape_luxor(session, limit=LIMIT_LUXOR)
print("Luxor done:", luxor_csv)


In [ ]:
# ---------------------------------------------------------------------------
# BB WHEELS — Wire Wheels only
# Phase 1: XML sitemap candidate URLs (replaces tire pagination from Cell 0)
# Phase 2: product page parse (same selectors as original Cell 1) + images
# Optional: USE_SELENIUM = True to also crawl Searchspring search pages
# ---------------------------------------------------------------------------
LIMIT_BB = None       # e.g. 20 for testing
USE_SELENIUM = False  # set True if selenium + chromedriver are installed

bb_csv = scrape_bbwheels(session, use_selenium=USE_SELENIUM, limit=LIMIT_BB)
print("BB done:", bb_csv)


In [ ]:
# Preview results
import pandas as pd
from pathlib import Path

for path in (OUTPUT_LUXOR, OUTPUT_BB):
    print("=" * 70)
    print(path.name, "exists=", path.exists())
    if path.exists():
        df = pd.read_csv(path)
        print("rows:", len(df), "cols:", len(df.columns))
        display_cols = [c for c in ["Product Name", "SKU", "Brand", "Size", "Sale Price", "Product URL", "Image1"] if c in df.columns]
        print(df[display_cols].head(10).to_string(index=False))


In [ ]:
# (Optional) Original-style one-off BB product parse demo
# Kept for reference — same BeautifulSoup approach as your Cell 1
import requests
from bs4 import BeautifulSoup

demo_url = "https://www.bbwheelsonline.com/american-racing-classic-torq-thrust-ii-one-piece-wheels-rims-15x7-5x127-5x5-gray-6-mm-offset-VN2155773/"
r = session.get(demo_url, timeout=40)
soup = BeautifulSoup(r.content, "html.parser")

name = soup.find("h1", class_="productView-title")
brand = soup.find("h2", class_="productView-brand")
price = soup.find("span", class_="price price--withoutTax")
print("Name:", name.get_text(strip=True) if name else None)
print("Brand:", brand.get_text(strip=True) if brand else None)
print("Price:", price.get_text(strip=True) if price else None)

details = {}
dl = soup.find("dl", class_="productView-info")
if dl:
    for dt, dd in zip(dl.find_all("dt"), dl.find_all("dd")):
        details[dt.get_text(strip=True).rstrip(":")] = dd.get_text(strip=True)
print("Sample details:", {k: details[k] for k in list(details)[:8]})
